# nb_admin__deploy_fabric_items

Deploys Fabric items from a client GitHub repo to a target prod workspace
using `fabric-cicd`. Downloads the repo at runtime via the GitHub API —
no local clone or VS Code extension required.

## Before Running
1. Update **Step 2** with client-specific config values
2. Ensure the Service Principal has Member or Admin access on the target workspace
3. Ensure your user has Key Vault Secrets User role on the client Key Vault
4. Ensure the GitHub PAT is stored in Key Vault with Contents read access to the client repo
5. Run cells in order — verify path output at the end of Step 4 before proceeding

## Parameterization
If a `parameter.yml` exists in the root of `FABRIC_ITEMS_FOLDER` in the repo,
`fabric-cicd` applies it automatically during deployment. No changes to this
notebook are needed — parameterization is fully driven by the client repo.

Use `$workspace.id` and `$items.<type>.<name>.id` as dynamic replace values
to avoid hardcoding prod GUIDs. Example:

```yaml
find_replace:
  - find_value: "dev-workspace-guid"
    replace_value:
      prod: "$workspace.id"
  - find_value: "dev-lakehouse-guid"
    replace_value:
      prod: "$items.Lakehouse.staging_lakehouse.id"
```

For first-time deployment to a new client, run without `parameter.yml` first
to create items in prod, then inspect item definition files in the repo for
embedded dev GUIDs, create `parameter.yml` in `fabric_workspace_items/`, and redeploy.
After initial setup, `parameter.yml` only needs updating when new item types
introducing new dev GUIDs are added to the workspace.

## What Gets Deployed
Controlled by `ITEM_TYPES_IN_SCOPE` in Step 2. Defaults:
- Notebooks
- Data Pipelines
- Lakehouses (shell only — no data)
- Warehouses (shell only — DDL managed separately via dbt)

## Notes
- Existing items in the target workspace are updated in place
- Items in prod not found in the repo are removed by `unpublish_all_orphan_items`
- `/tmp` storage is session-scoped and wiped when the session ends

## Step 1: Install Dependencies

Installs `fabric-cicd` into the current notebook session.
Required each session as packages do not persist between runs.
Always installs the latest version — pin a version if stability is required
e.g. `fabric-cicd==1.0.0`.

In [ ]:
%pip install fabric-cicd --quiet

## Step 2: Configure Deployment Parameters

Update these variables for each client before running:

- `KEY_VAULT_URL` — client Key Vault URL. SP credentials and GitHub PAT pulled from here
- `GITHUB_ORG` / `GITHUB_REPO` / `GITHUB_BRANCH` — client repo details
- `FABRIC_ITEMS_FOLDER` — must match exactly the Git folder configured in
  Workspace Settings → Git Integration. `parameter.yml` must be placed in
  the root of this folder in the repo if parameterization is needed
- `TARGET_WORKSPACE_ID` — GUID of the prod workspace, found in the workspace URL
- `ENVIRONMENT` — must match the environment key used in `parameter.yml`
  e.g. `prod`. If no `parameter.yml` exists yet this value is ignored
- `ITEM_TYPES_IN_SCOPE` — controls which item types get deployed.
  Comment out any types to skip for a given run

In [ ]:
# ── CLIENT CONFIG ── update these for each client ──────────────────────────

# Key Vault — SP credentials and GitHub PAT are stored here
KEY_VAULT_URL = "https://your-client-keyvault.vault.azure.net/"

# GitHub repo config — PAT pulled from Key Vault in Step 3
GITHUB_ORG    = "exscapegroup"
GITHUB_REPO   = "data-engineering"
GITHUB_BRANCH = "main"

# The Git folder configured in Workspace Settings → Git Integration
# parameter.yml must live in the root of this folder if parameterization is needed
FABRIC_ITEMS_FOLDER = "fabric_workspace_items"

# Target prod workspace
TARGET_WORKSPACE_ID = "your-prod-workspace-guid"
ENVIRONMENT         = "prod"  # must match environment key in parameter.yml

# Item types to deploy — comment out any to skip
ITEM_TYPES_IN_SCOPE = [
    "Notebook",
    "DataPipeline",
    "Lakehouse",
    "Warehouse",
]

## Step 3: Pull Credentials from Key Vault

Retrieves Service Principal credentials and GitHub PAT from Key Vault.
Your user must have Key Vault Secrets User role on the vault.

Expected secret names:
- `fabric-tenant-id` — Entra tenant ID
- `fabric-client-id` — Service Principal application (client) ID
- `fabric-client-secret` — Service Principal client secret
- `github-pat-fabric-integration` — fine-grained PAT with Contents read access to the repo

In [ ]:
import notebookutils

tenant_id     = notebookutils.credentials.getSecret(KEY_VAULT_URL, "fabric-tenant-id")
client_id     = notebookutils.credentials.getSecret(KEY_VAULT_URL, "fabric-client-id")
client_secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, "fabric-client-secret")
github_pat    = notebookutils.credentials.getSecret(KEY_VAULT_URL, "github-pat-fabric-integration")

print("Credentials retrieved successfully")

## Step 4: Download Repo from GitHub

Downloads the client repo as a zip via the GitHub API and extracts it
to `/tmp/fabric-deploy` in session storage.

- `/tmp` is session-scoped — wiped when the session ends, no cleanup needed
- GitHub zips extract into a dynamically named subdirectory so the root
  is located programmatically rather than hardcoded
- `resp.raise_for_status()` fails early if the PAT is invalid or the repo
  is inaccessible, before any deployment logic runs
- Verifies `FABRIC_ITEMS_FOLDER` exists in the extracted repo before continuing
- Checks for `parameter.yml` and reports status — deployment proceeds either way

In [ ]:
import requests
import zipfile
import io
import os
import glob

REPO_LOCAL_PATH = "/tmp/fabric-deploy"

def download_repo(org, repo, branch, pat, extract_path):
    url = f"https://api.github.com/repos/{org}/{repo}/zipball/{branch}"
    headers = {
        "Authorization": f"Bearer {pat}",
        "Accept": "application/vnd.github+json"
    }

    print(f"Downloading {org}/{repo} @ {branch}...")
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()

    z = zipfile.ZipFile(io.BytesIO(resp.content))
    z.extractall(extract_path)

    # GitHub zips extract into a subdirectory named "org-repo-commithash/"
    # locate it dynamically rather than hardcoding the path
    extracted_dirs = [
        d for d in glob.glob(f"{extract_path}/*/")
        if os.path.isdir(d)
    ]

    if not extracted_dirs:
        raise Exception("Could not find extracted repo directory")

    repo_root = extracted_dirs[0]
    print(f"Extracted to: {repo_root}")
    return repo_root

repo_path = download_repo(
    GITHUB_ORG,
    GITHUB_REPO,
    GITHUB_BRANCH,
    github_pat,
    REPO_LOCAL_PATH
)

# Resolve the full path to the Fabric items folder
items_dir = os.path.join(repo_path, FABRIC_ITEMS_FOLDER)

# Check for parameter.yml — informational only, deployment proceeds either way
param_file   = os.path.join(items_dir, "parameter.yml")
param_exists = os.path.exists(param_file)

print(f"Items directory    : {items_dir}")
print(f"Directory exists   : {os.path.exists(items_dir)}")
print(f"parameter.yml found: {param_exists}")

if not os.path.exists(items_dir):
    raise Exception(
        f"Fabric items folder '{FABRIC_ITEMS_FOLDER}' not found in repo. "
        f"Check FABRIC_ITEMS_FOLDER matches the Git folder in workspace settings."
    )

if not param_exists:
    print()
    print("WARNING: No parameter.yml found. Items will be deployed with dev GUIDs.")
    print("This is expected for a first deployment run.")
    print("After deployment, inspect item definition files in the repo for")
    print("embedded dev GUIDs and add parameter.yml to fabric_workspace_items/.")

## Step 5: Run the Deployment

Authenticates as the Service Principal and deploys all items in scope
to the target prod workspace.

If `parameter.yml` exists in the items folder, `fabric-cicd` applies it
automatically — no additional configuration needed here.

`unpublish_all_orphan_items` removes items from prod that no longer exist
in the repo, keeping prod in sync with the source branch.
Comment it out if you want an additive-only run.

In [ ]:
from azure.identity import ClientSecretCredential
from fabric_cicd import FabricWorkspace, publish_all_items, unpublish_all_orphan_items

token_credential = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

target_workspace = FabricWorkspace(
    workspace_id         = TARGET_WORKSPACE_ID,
    environment          = ENVIRONMENT,
    repository_directory = items_dir,
    item_type_in_scope   = ITEM_TYPES_IN_SCOPE,
    token_credential     = token_credential
)

print(f"Deploying to workspace : {TARGET_WORKSPACE_ID}")
print(f"Environment            : {ENVIRONMENT}")
print(f"Items in scope         : {ITEM_TYPES_IN_SCOPE}")
print(f"Parameterization       : {'enabled' if param_exists else 'skipped — no parameter.yml found'}")
print()

print("Starting deployment...")
publish_all_items(target_workspace)

print("Removing orphaned items from prod...")
unpublish_all_orphan_items(target_workspace)

print()
print("Deployment complete.")